# Stacking Transformer Blocks Lab

Stack several pre-norm blocks and verify that every layer preserves shape and supports a finite backward pass.


In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(15)

class Block(nn.Module):
    def __init__(self,width=12,heads=3):
        super().__init__()
        if width % heads: raise ValueError("width must be divisible by heads")
        self.h=heads; self.d=width//heads
        self.qkv=nn.Linear(width,3*width,bias=False)
        self.proj=nn.Linear(width,width,bias=False)
        self.ln1=nn.LayerNorm(width); self.ln2=nn.LayerNorm(width)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
    def attn(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).chunk(3,-1)
        def h(z): return z.view(B,T,self.h,self.d).transpose(1,2)
        q,k,v=h(q),h(k),h(v)
        s=q@k.transpose(-2,-1)/math.sqrt(self.d)
        m=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),diagonal=1)
        w=torch.softmax(s.masked_fill(m,float("-inf")),dim=-1)
        return self.proj((w@v).transpose(1,2).contiguous().view(B,T,C))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        return x+self.ff(self.ln2(x))

class Stack(nn.Module):
    def __init__(self,depth=4,width=12,heads=3):
        super().__init__()
        self.blocks=nn.ModuleList([Block(width,heads) for _ in range(depth)])
    def forward(self,x):
        shapes=[]
        for block in self.blocks:
            x=block(x); shapes.append(tuple(x.shape))
        return x,shapes

x=torch.randn(2,6,12,requires_grad=True)
model=Stack(depth=4)
out,shapes=model(x)
assert all(s==(2,6,12) for s in shapes)
assert out.shape==x.shape
out.mean().backward()
assert torch.isfinite(x.grad).all()
print("PASS: l05-15 stacked block checks")
